# 05 — Main Evaluation: Gemma-3n-E2B-IT (cross-model comparison)

Optional. Run if Qwen results look interesting and we want to test whether the confabulation finding generalizes across model families.

**Runtime:** ~1hr on H100 (Gemma-3n-E2B is smaller than Qwen2.5-Omni-7B).

**Caveat:** Gemma-3n's audio support is more limited than Qwen2.5-Omni's. If Gemma doesn't accept audio-only input cleanly (S1), we may need to skip S1 for Gemma and report findings only across S2–S5.


In [ ]:
import os, sys, json, time
REPO = '/content/omnimodel-research'
if os.path.exists(REPO):
    %cd $REPO
    sys.path.insert(0, REPO)

with open('data/eval_samples_clean.json') as f:
    samples = json.load(f)
print(f'Eval samples: {len(samples)}')


In [ ]:
from src.model_utils import GemmaOmniWrapper, TextOnlyModelWrapper
from src.prompts import STAGE_BUILDERS
from src.parse_utils import parse_answer_confidence, parse_attribution, parse_reason
from src.transcript_utils import load_transcript

gemma = GemmaOmniWrapper('google/gemma-3n-E2B-it')
text_only = TextOnlyModelWrapper('Qwen/Qwen2.5-1.5B-Instruct')

# Reuse stage_inputs and run_one from notebook 04
def stage_inputs(stage, sample):
    vid = sample['video_id']
    paths = {
        'video':      f'data/videos/{vid}.mp4',
        'audio':      f'data/audio/{vid}.wav',
        'silent':     f'data/silent_video/{vid}_silent.mp4',
        'transcript': f'data/transcripts/{vid}.txt',
        # Mismatched transcript is keyed by qa_id (a single video may have
        # multiple QAs that need DIFFERENT mismatched donors)
        'mismatched': f'data/transcripts_mismatched/{sample["qa_id"]}.txt',
    }
    routes = {
        'S0': dict(video_path=None, audio_path=None, extra={}),
        'S1': dict(video_path=None, audio_path=paths['audio'], extra={}),
        'S2': dict(video_path=paths['silent'], audio_path=None, extra={}),
        'S3': dict(video_path=paths['video'], audio_path=paths['audio'], extra={}),
        'S4': dict(video_path=paths['video'], audio_path=paths['audio'],
                   extra={'transcript': load_transcript(paths['transcript'])}),
        'S5': dict(video_path=paths['video'], audio_path=paths['audio'],
                   extra={'mismatched_transcript': load_transcript(paths['mismatched'])}),
    }
    return routes[stage]

def run_one(stage, sample, model):
    inp = stage_inputs(stage, sample)
    prompt = STAGE_BUILDERS[stage](sample['question'], sample['options'], **inp['extra'])
    resp = model.generate(prompt, video_path=inp['video_path'], audio_path=inp['audio_path'])
    answer, conf = parse_answer_confidence(resp.text)
    attr = parse_attribution(resp.text) if stage == 'S3' else None
    reason = parse_reason(resp.text) if stage == 'S3' else None
    return {
        'video_id': sample['video_id'], 'task_type': sample['task_type'], 'stage': stage,
        'ground_truth': sample['answer'], 'predicted_answer': answer,
        'verbalized_confidence': conf, 'answer_logprobs': resp.answer_logprobs,
        'attribution': attr, 'attribution_reason': reason, 'raw_response': resp.text,
    }

OUT_DIR = 'results/raw_predictions/gemma'
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
def run_stage(stage, samples, model, checkpoint_every=20):
    out_path = f'{OUT_DIR}/{stage}.json'
    done = []
    if os.path.exists(out_path):
        with open(out_path) as f:
            done = json.load(f)
    done_ids = {r['qa_id'] for r in done}
    todo = [s for s in samples if s['qa_id'] not in done_ids]
    if not todo:
        return done
    records = list(done)
    t0 = time.time()
    for i, s in enumerate(todo):
        try:
            rec = run_one(stage, s, model)
        except Exception as e:
            rec = {'video_id': s['video_id'], 'task_type': s['task_type'], 'stage': stage,
                   'ground_truth': s['answer'], 'predicted_answer': None,
                   'verbalized_confidence': None, 'answer_logprobs': None,
                   'attribution': None, 'attribution_reason': None,
                   'raw_response': f'<ERROR: {e}>'}
        records.append(rec)
        if (i + 1) % checkpoint_every == 0 or (i + 1) == len(todo):
            with open(out_path, 'w') as f:
                json.dump(records, f, indent=2)
            print(f'  {stage}: {len(records)}/{len(samples)}  ({(time.time()-t0)/(i+1):.1f}s/sample)')
    return records

# Skip S1 if audio-only doesn't work; we'll detect this empirically
STAGES = ['S0', 'S2', 'S3', 'S4', 'S5']  # try without S1 first
print('Running Gemma-3n stages: ', STAGES)
for stage in STAGES:
    print(f'\n=== {stage} ===')
    model = text_only if stage == 'S0' else gemma
    run_stage(stage, samples, model)


**Done.** Proceed to `06_analysis.ipynb`.